### Code Overview

The code is written in Python using PySpark to handle data processing in a Microsoft Fabric notebook. It retrieves parameters passed from the pipeline, creates a structured DataFrame, and saves the data to both a Lakehouse table and a CSV file. The code includes robust error handling and logging to ensure reliability.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import logging
try:
    from dbutils import dbutils
except ImportError:
    dbutils = None

# Initialize logging
logging.basicConfig(level=logging.INFO)

# Initialize Spark session
spark = SparkSession.builder.appName("ProcessCopyActivityParameters").getOrCreate()

# Function to get parameter with fallback methods
def get_parameter(param_name, default_value=''):
    # Method 1: Try global variable (Fabric injects as variables)
    try:
        value = globals()[param_name]
        logging.info(f"Retrieved {param_name} from global variables")
        return value
    except KeyError:
        logging.debug(f"Global variable {param_name} not found")

    # Method 2: Try spark.conf.get
    try:
        spark_param = f"spark.databricks.notebook.parameters.{param_name}"
        value = spark.conf.get(spark_param)
        logging.info(f"Retrieved {param_name} from spark.conf: {spark_param}")
        return value
    except Exception as e:
        logging.debug(f"spark.conf.get failed for {param_name}: {e}")

    # Method 3: Try dbutils.widgets.get
    if dbutils:
        try:
            value = dbutils.widgets.get(param_name)
            logging.info(f"Retrieved {param_name} from dbutils.widgets")
            return value
        except Exception as e:
            logging.debug(f"dbutils.widgets.get failed for {param_name}: {e}")

    # Method 4: Fallback to default
    logging.warning(f"Could not retrieve {param_name}, using default: {default_value}")
    return default_value

# Debug: Log global variables and Spark configurations
try:
    global_vars = [k for k in globals().keys() if not k.startswith('__')]
    logging.info(f"Global variables: {global_vars}")
    spark_configs = [conf for conf in spark.sparkContext.getConf().getAll() if 'notebook.parameters' in conf[0]]
    logging.info(f"Spark configurations with notebook.parameters: {spark_configs}")
except Exception as e:
    logging.error(f"Debug logging failed: {e}")

# Access parameters
pipeline_name = get_parameter('PipelineName', 'Unknown')
run_id = get_parameter('RunId', '')
execution_status = get_parameter('ExecutionStatus', 'Unknown')
execution_start = get_parameter('ExecutionStart', '')
execution_duration = get_parameter('ExecutionDuration', '0')
source_query = get_parameter('SourceQuery', '')
target_table = get_parameter('TargetTable', '')
rows_read = get_parameter('RowsRead', '0')
rows_written = get_parameter('RowsWritten', '0')
error_message = get_parameter('ErrorMessage', '')
timestamp = get_parameter('Timestamp', '')
data = get_parameter('Data', '{}')

# Convert execution_duration to integer, handle invalid values
try:
    duration = int(execution_duration)
except (ValueError, TypeError):
    logging.warning(f"Invalid ExecutionDuration: {execution_duration}, defaulting to 0")
    duration = 0

# Convert rows_read to integer, handle invalid values
try:
    rows_read_int = int(rows_read)
except (ValueError, TypeError):
    logging.warning(f"Invalid RowsRead: {rows_read}, defaulting to 0")
    rows_read_int = 0

# Convert rows_written to integer, handle invalid values
try:
    rows_written_int = int(rows_written)
except (ValueError, TypeError):
    logging.warning(f"Invalid RowsWritten: {rows_written}, defaulting to 0")
    rows_written_int = 0

# Log received parameters
logging.info(f"Parameters: PipelineName={pipeline_name}, RunId={run_id}, ExecutionStatus={execution_status}, ExecutionStart={execution_start}, ExecutionDuration={execution_duration}, SourceQuery={source_query}, TargetTable={target_table}, RowsRead={rows_read}, RowsWritten={rows_written}, ErrorMessage={error_message}, Timestamp={timestamp}, Data={data}")

# Define schema for the DataFrame
schema = StructType([
    StructField("PipelineName", StringType(), True),
    StructField("RunId", StringType(), True),
    StructField("ExecutionStatus", StringType(), True),
    StructField("ExecutionStart", StringType(), True),
    StructField("ExecutionDuration", IntegerType(), True),
    StructField("SourceQuery", StringType(), True),
    StructField("TargetTable", StringType(), True),
    StructField("RowsRead", IntegerType(), True),
    StructField("RowsWritten", IntegerType(), True),
    StructField("ErrorMessage", StringType(), True),
    StructField("Timestamp", StringType(), True),
    StructField("Data", StringType(), True)
])

# Create a single-row DataFrame
data_row = [(
    pipeline_name,
    run_id,
    execution_status,
    execution_start,
    duration,
    source_query,
    target_table,
    rows_read_int,
    rows_written_int,
    error_message,
    timestamp,
    data
)]
df = spark.createDataFrame(data_row, schema)

# Display the DataFrame
display(df)

# Save to a Lakehouse table
try:
    df.write.mode("append").saveAsTable("ActivityDetailLogs")
    logging.info("Saved to Lakehouse table: ActivityDetailLogs")
except Exception as e:
    logging.error(f"Failed to save to Lakehouse table: {e}")

# Save to Files section as CSV
output_path = f"Files/Logs/ActivityLogs/ActivityLog_{run_id or 'unknown'}_{timestamp.replace(':', '-')}.csv"
try:
    df.write.csv(output_path, header=True, mode="overwrite")
    logging.info(f"Saved to Files section: {output_path}")
except Exception as e:
    logging.error(f"Failed to save to Files section: {e}")

# Log a summary
print(f"Processed Copy Activity Parameters: Pipeline={pipeline_name}, RunId={run_id}, Table={target_table}, Status={execution_status}, RowsRead={rows_read_int}, RowsWritten={rows_written_int}, Timestamp={timestamp}")

